<a href="https://colab.research.google.com/github/engtasneemalassaf-tech/Clinical-Symptom-Screening-Assistant-for-Nurses/blob/main/Clinical_Symptom_Screening_Assistant_(for_Nurses)22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🩺 Medical Screening Assistant  
**Chatbot to Help Nurses in Early Patient Triage**  
  

**Objective:** Build a nurse-friendly prototype covering all rubric items at Distinction level.  
**Models:**  
- Model A: Logistic Regression (sklearn)  
- Model B: Flan-T5-base (fast, CPU-friendly, public)  

**Datasets:**  
- Main: itachi9604/disease-symptom-description-dataset  
- Secondary: uom190346a/disease-symptoms-and-patient-profile-dataset  

**Final UI:** Clean, professional, nurse-focused (symptom input, extracted symptoms, Model A top predictions, Model B guidance, escalation note).

# **1.Install Dependencies**

In [ ]:
!pip install -q gradio==4.42.0 pandas scikit-learn fuzzywuzzy python-Levenshtein kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 1.60.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 12.0 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 12.0 which is incompatible.
dataproc-spark-connect 1.0.1 requires websockets>=14.0, but you have websockets 12.0 which is incompatible.
google-adk 1.23.0 requires websockets<16

# **2.Download Datasets (kagglehub)**

In [ ]:
import kagglehub
import os

print("Downloading main dataset...")
path1 = kagglehub.dataset_download("itachi9604/disease-symptom-description-dataset")
print("Dataset 1 path:", path1)

print("Downloading secondary dataset...")
path2 = kagglehub.dataset_download("uom190346a/disease-symptoms-and-patient-profile-dataset")
print("Dataset 2 path:", path2)

print("Files in main dataset:", os.listdir(path1))
print("Files in secondary dataset:", os.listdir(path2))

100%|██████████| 30.1k/30.1k [00:00<00:00, 24.3MB/s]

Extracting files...
Dataset 1 path: /root/.cache/kagglehub/datasets/itachi9604/disease-symptom-description-dataset/versions/2


100%|██████████| 3.07k/3.07k [00:00<00:00, 6.22MB/s]

Extracting files...
Dataset 2 path: /root/.cache/kagglehub/datasets/uom190346a/disease-symptoms-and-patient-profile-dataset/versions/2
Files in main dataset: ['symptom_precaution.csv', 'Symptom-severity.csv', 'dataset.csv', 'symptom_Description.csv']
Files in secondary dataset: ['Disease_symptom_and_patient_profile_dataset.csv']


# **3.Load Data & Create Lookup Maps**

In [ ]:
import pandas as pd

data_file = os.path.join(path1, "dataset.csv")
desc_file = os.path.join(path1, "symptom_Description.csv")
prec_file = os.path.join(path1, "symptom_precaution.csv")

df = pd.read_csv(data_file)
desc_df = pd.read_csv(desc_file)
prec_df = pd.read_csv(prec_file)

desc_map = dict(zip(desc_df["Disease"].str.strip(), desc_df["Description"].str.strip()))

prec_map = {}
for _, row in prec_df.iterrows():
    d = str(row["Disease"]).strip()
    p = [str(row[c]).strip() for c in prec_df.columns if "precaution" in c.lower() and pd.notna(row[c])]
    prec_map[d] = ", ".join(p)

print(f"Loaded {len(df)} records, {len(desc_map)} diseases, {len(prec_map)} precautions")

Loaded 4920 records, 41 diseases, 41 precautions


# **4.Train Model A (Logistic Regression)**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Print columns of df to debug the KeyError
print("Columns in df:", df.columns.tolist())

target_col = "Disease" # This has been corrected based on actual df columns
X_raw = df.drop(columns=[target_col])
y_raw = df[target_col].astype(str)

symptom_vocab = sorted(set(
    str(s).strip().lower().replace(" ", "_")
    for col in X_raw.columns
    for s in X_raw[col].dropna().astype(str)
    if str(s).strip().lower() not in ["", "nan"]
))
symptom_index = {s: i for i, s in enumerate(symptom_vocab)}

def row_to_multihot(row):
    vec = np.zeros(len(symptom_vocab), dtype=int)
    for val in row:
        if pd.isna(val): continue
        s = str(val).strip().lower().replace(" ", "_")
        if s in symptom_index:
            vec[symptom_index[s]] = 1
    return vec

X = np.array([row_to_multihot(r) for _, r in X_raw.iterrows()])
le = LabelEncoder()
y = le.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lr = LogisticRegression(max_iter=2000, n_jobs=-1)
lr.fit(X_train, y_train)

acc = lr.score(X_test, y_test)
print(f"Model A (LR) test accuracy: {acc:.4f} ({acc*100:.2f}%)")

Columns in df: ['Disease', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Symptom_4', 'Symptom_5', 'Symptom_6', 'Symptom_7', 'Symptom_8', 'Symptom_9', 'Symptom_10', 'Symptom_11', 'Symptom_12', 'Symptom_13', 'Symptom_14', 'Symptom_15', 'Symptom_16', 'Symptom_17']
Model A (LR) test accuracy: 1.0000 (100.00%)


# **5.Load Model B (Flan-T5-base – CPU friendly)**

In [ ]:
from transformers import pipeline

pipe = pipeline("text2text-generation", model="google/flan-t5-base", device=-1)
print("Model B loaded (fast on CPU)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


Model B loaded (fast on CPU)


# **6.Core Functions (Extraction, Prompt, Escalation)**

In [ ]:
import re
from fuzzywuzzy import process

DISCLAIMER = "This tool is for educational purposes only and should not be used as a substitute for professional medical advice. Always consult with a qualified healthcare provider for any health concerns."

def extract_symptoms(text):
    t = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    t = re.sub(r"\s+", " ", t).strip()
    found = []
    for s in symptom_vocab:
        if s.replace("_", " ") in t:
            found.append(s)
    for w in t.split():
        if len(w) > 4:
            match, score = process.extractOne(w, [s.replace("_", " ") for s in symptom_vocab])
            if score >= 88:
                found.append(match.replace(" ", "_"))
    return sorted(set(found))

def run_ui(user_text):
    symptoms = extract_symptoms(user_text)
    symptoms_str = ", ".join(symptoms) if symptoms else "none"

    if not symptoms:
        return symptoms_str, "(no match)", "Describe symptoms clearly.", "No escalation"

    vec = np.zeros(len(symptom_vocab))
    for s in symptoms:
        if s in symptom_index:
            vec[symptom_index[s]] = 1
    vec = vec.reshape(1, -1)

    probs = lr.predict_proba(vec)[0]
    top_idx = np.argsort(probs)[::-1][:5]
    top_pairs = [(le.inverse_transform([i])[0], float(probs[i])) for i in top_idx]

    a_out = ""
    for rank, (d, p) in enumerate(top_pairs, 1):
        if p < 0.05: continue
        desc = desc_map.get(d, "No desc")[:300] + "..."
        prec = prec_map.get(d, "No prec")[:200] + "..."
        a_out += f"{rank}. {d} (Conf: {p:.1%})\nDesc: {desc}\nPrec: {prec}\n---\n"

    top_d_str = ", ".join([d for d,p in top_pairs[:3]])
    prompt = f"""You are a nurse triage assistant. Symptoms: {symptoms_str}. Reference ML: {top_d_str}. Give safe advice, suggest nurse actions, end with disclaimer."""
    b_out = pipe(prompt, max_length=250)[0]["generated_text"].strip()
    b_out += "\n\n" + DISCLAIMER

    escal = []
    if top_pairs[0][1] < 0.35: escal.append("Low confidence")
    if len(symptoms) < 3: escal.append("Few symptoms")
    if any(w in user_text.lower() for w in ["chest pain", "difficulty breathing"]): escal.append("Emergency")

    note = "**ESCALATE**:\n" + "\n".join(escal) if escal else "OK"
    note += "\n\n" + DISCLAIMER

    return symptoms_str, a_out, b_out, note

# **RAG**

In [ ]:
!pip -q install -U langchain langchain-community langchain-text-splitters

!pip -q install -U faiss-cpu sentence-transformers

!pip -q install -U langchain-openai tiktoken python-dotenv

!pip -q install -U rouge-score

!pip install pypdf

!pip -q install gradio


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:

import os
import shutil

os.makedirs("data", exist_ok=True)

for file in os.listdir():
    if file.endswith(".pdf"):
        shutil.move(file, "data/" + file)

print("Files inside data:", os.listdir("data"))

Files inside data: ['The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf']


In [ ]:

import os
from langchain_community.document_loaders import PyPDFLoader

DATA_PATH = "data/"
documents = []

for file_name in os.listdir(DATA_PATH):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(DATA_PATH, file_name)
        loader = PyPDFLoader(file_path)
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages from PDFs")

Loaded 759 pages from PDFs


In [ ]:
# ────────────────────────────────────────────────
# Medical-aware chunking (special processing before splitting)
# ────────────────────────────────────────────────
import re
from typing import List, Dict
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# ─── 1. Medical dictionaries (expand with terms from Gale or your knowledge) ───
abbreviation_expansion_map = {
    r"\bMVC\b": "motor vehicle collision",
    r"\bMI\b": "myocardial infarction",
    r"\bCVA\b": "cerebrovascular accident",
    r"\bSOB\b": "shortness of breath",
    r"\bHTN\b": "hypertension",
    r"\bDM\b": "diabetes mellitus",
    r"\bCOPD\b": "chronic obstructive pulmonary disease",
    r"\bCHF\b": "congestive heart failure",
    r"\bDVT\b": "deep vein thrombosis",
    r"\bPE\b": "pulmonary embolism",
    r"\bUTI\b": "urinary tract infection",
    r"\bCBC\b": "complete blood count",
    r"\bBP\b": "blood pressure",
    r"\bHR\b": "heart rate",
    r"\bRR\b": "respiratory rate",
    # Add more common abbreviations found in Gale Encyclopedia
}

protected_medical_phrases = [
    "acute myocardial infarction",
    "chronic obstructive pulmonary disease",
    "congestive heart failure",
    "deep vein thrombosis",
    "pulmonary embolism",
    "urinary tract infection",
    "motor vehicle collision",
    "shortness of breath",
    "chest pain",
    "ibuprofen 200 mg",
    "aspirin 81 mg",
    "blood pressure",
    # Add more common multi-word terms / drug names / disease names from the book
]

# ─── 2. Processing functions ───
def clean_medical_text(text: str) -> str:
    """Remove common PDF noise found in books like Gale Encyclopedia"""
    text = re.sub(r'Page \d+ of \d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    text = re.sub(r'\s+', ' ', text.strip())
    return text


def expand_abbreviations(text: str) -> str:
    for pattern, full_form in abbreviation_expansion_map.items():
        text = re.sub(pattern, full_form, text, flags=re.IGNORECASE)
    return text


def protect_compound_phrases(text: str, phrases: List[str]) -> tuple[str, Dict[str, str]]:
    """Replace multi-word medical phrases with placeholders to prevent splitting"""
    placeholder_map = {}
    for i, phrase in enumerate(phrases):
        placeholder = f"__PHRASE_{i:04d}__"
        placeholder_map[placeholder] = phrase
        text = re.sub(re.escape(phrase), placeholder, text, flags=re.IGNORECASE)
    return text, placeholder_map


def restore_phrases(text: str, placeholder_map: Dict[str, str]) -> str:
    for placeholder, original_phrase in placeholder_map.items():
        text = text.replace(placeholder, original_phrase)
    return text


# ─── 3. Apply processing to documents (coming from PyPDFLoader.load()) ───
processed_documents = []

for doc in documents:  # documents = loader.load()
    content = doc.page_content

    content = clean_medical_text(content)
    content = expand_abbreviations(content)

    protected_content, phrase_placeholders = protect_compound_phrases(content, protected_medical_phrases)

    if len(protected_content.strip()) < 80:
        continue  # skip very small fragments

    # Create new document with temporarily protected content
    processed_doc = Document(
        page_content=protected_content,
        metadata=doc.metadata
    )
    processed_doc.metadata["original_phrases"] = phrase_placeholders  # store for later restoration

    processed_documents.append(processed_doc)


# ─── 4. Medical-aware splitting ───
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", "; ", " - ", " ", ""],
    keep_separator=True,
    add_start_index=True,           # useful later for citations
)

chunks = text_splitter.split_documents(processed_documents)


# ─── 5. Restore original phrases in every chunk ───
final_chunks = []
for chunk in chunks:
    restored_content = restore_phrases(
        chunk.page_content,
        chunk.metadata.get("original_phrases", {})
    )
    chunk.page_content = restored_content
    final_chunks.append(chunk)

print(f"Created {len(final_chunks)} medically-aware text chunks")

Created 8130 medically-aware text chunks


In [ ]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer


model = SentenceTransformer('all-MiniLM-L6-v2')

texts_to_encode = [chunk.page_content for chunk in chunks]
embeddings = model.encode(texts_to_encode, show_progress_bar=True)

print("Shape of embeddings:", embeddings.shape)

Batches:   0%|          | 0/255 [00:00<?, ?it/s]

Shape of embeddings: (8130, 384)


In [ ]:
import faiss
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [ ]:
import re
import numpy as np

def run_ui(user_text):
    # 1. Extract symptoms (same as before)
    symptoms = extract_symptoms(user_text)
    symptoms_str = ", ".join(symptoms) if symptoms else "none"

    if not symptoms:
        return symptoms_str, "(no match)", "Please describe symptoms more clearly.", "No escalation"

    # 2. Create query embedding
    query_embedding = model.encode([symptoms_str])[0].astype('float32')

    # 3. Search RAG (top 4 chunks)
    distances, indices = index.search(np.array([query_embedding]), k=4)

    # 4. Build context from retrieved chunks
    context_pieces = []
    for idx in indices[0]:
        if idx == -1: continue
        chunk_text = final_chunks[idx].page_content.strip()
        source = f"(from {final_chunks[idx].metadata.get('source', 'medical book')}, page ~{final_chunks[idx].metadata.get('page', '?')})"
        context_pieces.append(f"{chunk_text}\n{source}")

    context = "\n\n".join(context_pieces) if context_pieces else "No relevant information found."

    # 5. Model A output (top 5 probabilities – display only)
    vec = np.zeros(len(symptom_vocab))
    for s in symptoms:
        if s in symptom_index:
            vec[symptom_index[s]] = 1
    vec = vec.reshape(1, -1)
    probs = lr.predict_proba(vec)[0]
    top_idx = np.argsort(probs)[::-1][:5]
    top_pairs = [(le.inverse_transform([i])[0], float(probs[i])) for i in top_idx]

    a_out = ""
    for rank, (d, p) in enumerate(top_pairs, 1):
        if p < 0.05: continue
        color = "green" if p > 0.5 else "orange" if p > 0.2 else "red"
        desc = desc_map.get(d, "No description")[:300] + "..."
        prec = prec_map.get(d, "No precautions")[:200] + "..."
        a_out += f'<div style="color:{color}">{rank}. {d} (Conf: {p:.1%})</div>'
        a_out += f'Desc: {desc}\nPrec: {prec}\n---\n'

    # 6. Improved prompt with RAG context
    prompt = f"""You are a safe, conservative nursing assistant.
Do NOT name specific diseases unless absolutely necessary. Focus on symptoms and general actions.

Patient symptoms: {symptoms_str}

Reference information extracted from a trusted medical book:
{context}

Follow these steps:
1. Briefly summarize the main symptoms.
2. Identify possible general risks (e.g. infection, inflammation, breathing issue) without diagnosing.
3. Suggest immediate nursing actions (vitals, additional questions, monitoring, consult doctor).
4. State when immediate emergency escalation is needed.
5. Always end with: This is not a medical diagnosis.

Response:"""

    # 7. Generate Model B response
    b_out_raw = pipe(prompt, max_length=350, do_sample=False)[0]["generated_text"].strip()
    b_out = b_out_raw + "\n\n" + DISCLAIMER

    # 8. Escalation logic
    escal = []
    emergency_words = ["chest pain", "ألم صدر", "difficulty breathing", "ضيق تنفس", "shortness of breath"]
    if any(w in user_text.lower() for w in emergency_words):
        escal.append("Potential emergency – immediate emergency department recommended")
    if len(symptoms) <= 2 and "fever" not in symptoms_str.lower():
        escal.append("Few symptoms – further follow-up advised")
    if top_pairs and top_pairs[0][1] < 0.30:
        escal.append("Low classification confidence – medical evaluation recommended")

    note = "**Escalation recommendation**:\n" + "\n".join(escal) if escal else "Relatively stable – monitoring is appropriate"
    note += "\n\n" + DISCLAIMER

    return symptoms_str, a_out, b_out, note

# **7.Final Nurse-Friendly UI**

In [42]:
import gradio as gr
import numpy as np
import re

DISCLAIMER = (
    "This tool is for educational purposes only and should not be used as a substitute for professional medical advice. "
    "Always consult with a qualified healthcare provider for any health concerns."
)

# -----------------------------
# Helpers: safe getters
# -----------------------------
def _get_desc_prec(disease: str):
    # Prefer rag_map if exists (your old format)
    if "rag_map" in globals() and isinstance(rag_map, dict) and disease in rag_map:
        desc = (rag_map[disease].get("description") or "").strip()
        prec = (rag_map[disease].get("precautions") or "").strip()
        return desc, prec

    # Otherwise use desc_map / prec_map if available
    desc = (desc_map.get(disease) or "").strip() if "desc_map" in globals() else ""
    prec = (prec_map.get(disease) or "").strip() if "prec_map" in globals() else ""
    return desc, prec

def _rag_book_context(user_text, symptoms, k=3):
    if "rag_retrieve" not in globals():
        return ""
    try:
        q = f"triage nursing actions red flags general risks for: {', '.join(symptoms)}. patient: {user_text}"
        docs = rag_retrieve(q, k=k) or []
        if not docs:
            return ""
        parts = []
        for i, d in enumerate(docs, 1):
            txt = re.sub(r"\s+", " ", (getattr(d, "page_content", "") or "")).strip()
            parts.append(f"[BOOK{i}] {txt[:380]}")
        return "\n".join(parts)
    except Exception:
        return ""

# -----------------------------
# Model B: 4 lines ONLY + anti-repeat + safety
# -----------------------------
UNSAFE_PATTERNS = [
    r"\bpatient has\b", r"\byou have\b", r"\bdiagnos", r"\bconfirmed\b", r"\bdefinitely\b",
    r"\bmg\b", r"\bdose\b", r"\bdosage\b", r"\bantibiotic\b", r"\bibuprofen\b", r"\bparacetamol\b",
]

def _looks_bad(text: str) -> bool:
    if not text or len(text.strip()) < 60:
        return True
    low = text.lower()
    if any(re.search(p, low) for p in UNSAFE_PATTERNS):
        return True
    # repetition heuristic
    sents = [s.strip() for s in re.split(r"[.\n]+", text) if s.strip()]
    if len(sents) >= 6:
        uniq = len(set(sents))
        if uniq / len(sents) < 0.6:
            return True
    return False

def _fallback_b(symptoms_str, note):
    return (
        f"Main concern: Screening suggests an acute issue requiring basic assessment based on: {symptoms_str}.\n"
        "Nurse actions: Check Temp/HR/RR/BP/SpO2, assess onset/duration/severity, hydration, pain score; document.\n"
        "When to escalate: Any chest pain, shortness of breath, confusion, fainting, unstable vitals, or rapid deterioration.\n"
        "Next step: Follow protocol + re-assess; escalate for clinician review if uncertainty persists.\n\n"
        + DISCLAIMER
    )

def _generate_model_b_4lines(user_text, symptoms, top_pairs, note):
    symptoms_str = ", ".join(symptoms) if symptoms else "none detected"

    # Dataset grounding from top-2 candidates
    data_lines = []
    for disease, prob in top_pairs[:2]:
        desc, prec = _get_desc_prec(disease)
        desc = (desc[:240] if desc else "—")
        prec = (prec[:180] if prec else "—")
        data_lines.append(f"- {disease} (conf={prob:.2f}) | desc: {desc} | precautions: {prec}")
    data_ctx = "\n".join(data_lines) if data_lines else "No dataset candidates."

    # Book RAG grounding
    book_ctx = _rag_book_context(user_text, symptoms, k=3)
    if not book_ctx:
        book_ctx = "No book snippets."

    prompt = f"""
You are a nurse screening assistant. Provide a SHORT, ACTIONABLE response for a nurse under pressure.
Use ONLY the evidence below. Do NOT diagnose. Do NOT recommend medications/doses.
Ignore demographics (age/sex/race/nationality). Focus on symptoms and safety.

Return EXACTLY 4 lines, in this exact order and labels:
Main concern: ...
Nurse actions: ...
When to escalate: ...
Next step: ...

Patient text:
{user_text}

Extracted symptoms:
{symptoms_str}

Safety note:
{note}

Evidence (dataset candidates + precautions):
{data_ctx}

Evidence (book RAG snippets):
{book_ctx}
""".strip()

    # Generate with anti-repeat controls
    out = pipe(
        prompt,
        max_new_tokens=140,
        do_sample=False,
        repetition_penalty=1.20,
        no_repeat_ngram_size=4
    )[0].get("generated_text", "").strip()

    # Extract only the 4 required labeled lines
    lines = [l.strip() for l in out.splitlines() if l.strip()]
    wanted = ["main concern:", "nurse actions:", "when to escalate:", "next step:"]
    picked = []
    for w in wanted:
        for l in lines:
            if l.lower().startswith(w):
                picked.append(l)
                break

    final = "\n".join(picked) if len(picked) == 4 else out

    if _looks_bad(final):
        return _fallback_b(symptoms_str, note)

    if DISCLAIMER not in final:
        final += f"\n\n{DISCLAIMER}"

    return final

# -----------------------------
# Model A formatter (your style)
# Top1 red, others green, includes desc+prec
# -----------------------------
def _format_model_a(top_pairs):
    a_out = ""
    for rank, (disease, prob) in enumerate(top_pairs, 1):
        # top1 red, others green
        color = "#c62828" if rank == 1 else "#2e7d32"
        desc, prec = _get_desc_prec(disease)
        desc = (desc.strip()[:450] + "…") if desc and desc.strip() else "—"
        prec = (prec.strip()[:250] + "…") if prec and prec.strip() else "—"

        a_out += f'<div style="color:{color}; font-weight:900;">{rank}. 🩺 Disease: {disease}</div>'
        a_out += f'<div style="color:{color}; font-weight:800;">Confidence: {prob:.2f}</div>'
        a_out += f'<div><b>Description:</b> {desc}</div>'
        a_out += f'<div><b>Precautions:</b> {prec}</div>'
        a_out += '<div style="border-bottom:1px dashed #d1d5db; margin:10px 0;"></div>'
    return a_out or "(no predictions)"

# -----------------------------
# Escalation note
# -----------------------------
def _safety_note(user_text, symptoms, top_prob):
    escal_msgs = []
    red_flags = [
        "chest pain", "difficulty breathing", "shortness of breath",
        "unconscious", "seizure", "suicide", "self-harm"
    ]
    if any(rf in (user_text or "").lower() for rf in red_flags):
        escal_msgs.append("Potential emergency red-flag — escalate immediately per protocol")

    if not escal_msgs:
        if top_prob < 0.35:
            escal_msgs.append("Low confidence — gather more symptoms / clinician review")
        if len(symptoms) < 3:
            escal_msgs.append("Few symptoms — clarify onset/duration/severity and associated factors")

    if escal_msgs:
        return "ESCALATION RECOMMENDED:\n" + "\n".join(f"• {m}" for m in escal_msgs) + f"\n\n{DISCLAIMER}"
    return f"No immediate escalation. Monitor and reassess.\n\n{DISCLAIMER}"

# -----------------------------
# run_ui (safe, no Gradio ERROR)
# -----------------------------
def run_ui(user_text):
    try:
        # Check required objects exist
        required = ["extract_symptoms", "symptom_vocab", "symptom_index", "lr", "le", "pipe"]
        missing = [x for x in required if x not in globals()]
        if missing:
            msg = "Developer hint: missing -> " + ", ".join(missing)
            return "none detected", "(Model A unavailable)", msg, msg

        symptoms = extract_symptoms(user_text)
        symptoms_str = ", ".join(symptoms) if symptoms else "none detected"

        if not symptoms:
            note = "Few / no matched symptoms — add more details."
            b_out = _fallback_b(symptoms_str, note)
            return symptoms_str, "(no matching symptoms)", b_out, _safety_note(user_text, symptoms, 0.0)

        # Build vector
        vec = np.zeros(len(symptom_vocab), dtype=float)
        for s in symptoms:
            if s in symptom_index:
                vec[symptom_index[s]] = 1.0
        vec = vec.reshape(1, -1)

        # Predict
        probs = lr.predict_proba(vec)[0]
        top_idx = np.argsort(probs)[::-1][:5]
        top_pairs = [(le.inverse_transform([i])[0], float(probs[i])) for i in top_idx]

        top_prob = top_pairs[0][1] if top_pairs else 0.0
        note = _safety_note(user_text, symptoms, top_prob)

        # Model A output
        a_out = _format_model_a(top_pairs)

        # Model B output (4 lines only, grounded)
        b_out = _generate_model_b_4lines(user_text, symptoms, top_pairs, note)

        return symptoms_str, a_out, b_out, note

    except Exception as e:
        msg = f"Developer error: {type(e).__name__}: {e}"
        return "none detected", "(error)", msg, msg

# =======================================================
# Final UI – Clean & Professional (your style)
# =======================================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"), title="Nurse Screening Assistant") as demo:
    gr.Markdown("# 🩺 Medical Screening Assistant")
    gr.Markdown("**Educational prototype for nurses** – Symptom screening support only")

    gr.Markdown(
        f"<div style='background-color:#ffebee; color:#c62828; font-weight:bold; font-size:16px; padding:16px; "
        f"border-radius:8px; border:2px solid #c62828; margin-bottom:20px;'>{DISCLAIMER}</div>"
    )

    inp = gr.Textbox(
        lines=6,
        label="Patient Symptoms Description",
        placeholder="e.g. high fever for 4 days, dry cough, severe fatigue, muscle pain"
    )
    btn = gr.Button("Analyze", variant="primary")

    with gr.Row(equal_height=True):
        with gr.Column():
            out_sym = gr.Textbox(label="Extracted Symptoms", lines=3)
            out_a = gr.Markdown(label="Model A – Logistic Regression (Top Predictions)")
        with gr.Column():
            out_b = gr.Textbox(lines=8, label="Model B – Nurse Guidance (4-line)")
            out_note = gr.Textbox(label="Safety & Escalation Note", lines=6, interactive=False)

    btn.click(run_ui, inputs=inp, outputs=[out_sym, out_a, out_b, out_note])

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.42.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Running on public URL: https://cb595d28d2049d6cf5.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cb595d28d2049d6cf5.gradio.live


In [43]:
import gradio as gr
import numpy as np
import re

DISCLAIMER = (
    "This tool is for educational purposes only and should not be used as a substitute for professional medical advice. "
    "Always consult with a qualified healthcare provider for any health concerns."
)

# =========================================================
# 1) PROMPT STRATEGIES (4 BULLETS, NOT 4 LINES)
# =========================================================
PROMPT_S1_ROLE_BASIC = """You are a nurse triage assistant. Be concise and actionable.

Patient symptoms: {symptoms_str}

Return ONLY 4 bullet points with these exact headings (do not add extra sections):
- Main concern: ...
- Nurse actions: ...
- When to escalate: ...
- Next step: ...

Rules:
- Do NOT diagnose or name diseases.
- Do NOT recommend medications or dosages.
- Do NOT repeat sentences.
- Keep each bullet short.

Evidence (database/book):
{rag_context}

End with: DISCLAIMER: {disclaimer}
"""

PROMPT_S2_ROLE_RAG = """You are an experienced nurse triage assistant. Be concise and actionable.

Patient symptoms: {symptoms_str}
Context from database: {rag_context}

Return ONLY 4 bullet points with these exact headings (do not add extra sections):
- Main concern: ...
- Nurse actions: ...
- When to escalate: ...
- Next step: ...

Rules:
- Do NOT diagnose or name diseases.
- Do NOT recommend medications or dosages.
- Do NOT repeat sentences.
- Keep each bullet short.

End with: DISCLAIMER: {disclaimer}
"""

PROMPT_S3_SAFETY_BIAS = """You are a SAFE nurse triage assistant for screening only.

BIAS RULE: Ignore demographics (age/sex/nationality/race). Focus only on symptoms and safety.
SAFETY RULES:
- Do NOT diagnose or name diseases.
- Do NOT recommend medications or dosages.
- Do NOT repeat sentences.
- If evidence is weak, recommend escalation.

Patient symptoms: {symptoms_str}

Evidence (database/book):
{rag_context}

Return ONLY 4 bullet points with these exact headings (do not add extra sections):
- Main concern: ...
- Nurse actions: ...
- When to escalate: ...
- Next step: ...

Keep it short and helpful for a nurse under pressure.
End with: DISCLAIMER: {disclaimer}
"""

PROMPT_BANK = {
    "S1: Role + Basic": PROMPT_S1_ROLE_BASIC,
    "S2: Role + RAG": PROMPT_S2_ROLE_RAG,
    "S3: Safety + Bias (Best)": PROMPT_S3_SAFETY_BIAS,
}

# =========================================================
# 2) Helpers: safe getters
# =========================================================
def _get_desc_prec(disease: str):
    if "rag_map" in globals() and isinstance(rag_map, dict) and disease in rag_map:
        desc = (rag_map[disease].get("description") or "").strip()
        prec = (rag_map[disease].get("precautions") or "").strip()
        return desc, prec
    desc = (desc_map.get(disease) or "").strip() if "desc_map" in globals() else ""
    prec = (prec_map.get(disease) or "").strip() if "prec_map" in globals() else ""
    return desc, prec

def _rag_book_context(user_text, symptoms, k=3):
    if "rag_retrieve" not in globals():
        return ""
    try:
        q = f"triage nursing actions red flags general risks for: {', '.join(symptoms)}. patient: {user_text}"
        docs = rag_retrieve(q, k=k) or []
        if not docs:
            return ""
        parts = []
        for i, d in enumerate(docs, 1):
            txt = re.sub(r"\s+", " ", (getattr(d, "page_content", "") or "")).strip()
            parts.append(f"[BOOK{i}] {txt[:360]}")
        return "\n".join(parts)
    except Exception:
        return ""

# =========================================================
# 3) Model B: bullets only + anti-repeat + safe parsing
# =========================================================
UNSAFE_PATTERNS = [
    r"\bdiagnos", r"\bconfirmed\b", r"\bdefinitely\b",
    r"\bmg\b", r"\bdose\b", r"\bdosage\b",
    r"\bantibiotic\b", r"\bibuprofen\b", r"\bparacetamol\b",
]

def _looks_bad(text: str) -> bool:
    if not text or len(text.strip()) < 50:
        return True
    low = text.lower()
    if any(re.search(p, low) for p in UNSAFE_PATTERNS):
        return True
    sents = [s.strip() for s in re.split(r"[.\n]+", text) if s.strip()]
    if len(sents) >= 6:
        uniq = len(set(sents))
        if uniq / len(sents) < 0.6:
            return True
    return False

def _safe_format(template: str, symptoms_str: str, rag_context: str):
    symptoms_str = symptoms_str or "none detected"
    rag_context = rag_context or "No context retrieved."
    try:
        return template.format(
            symptoms_str=symptoms_str,
            rag_context=rag_context,
            disclaimer=DISCLAIMER
        )
    except Exception:
        # fallback replace (never crash)
        return (template
                .replace("{symptoms_str}", symptoms_str)
                .replace("{rag_context}", rag_context)
                .replace("{disclaimer}", DISCLAIMER))

def _pipe_generate(prompt: str) -> str:
    out = pipe(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        repetition_penalty=1.20,
        no_repeat_ngram_size=4
    )
    if isinstance(out, list) and out and isinstance(out[0], dict):
        return (out[0].get("generated_text") or out[0].get("text") or "").strip()
    return str(out).strip()

def _enforce_4_bullets(text: str) -> str:
    """
    Ensure exactly 4 bullets exist with correct headings.
    If model returns extra stuff, we pick the 4 bullets.
    """
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    # collect bullets starting with "-" or "•"
    bullets = [l for l in lines if l.startswith("-") or l.startswith("•")]

    # Normalize bullet prefix to "-"
    bullets = [("- " + b.lstrip("-• ").strip()) for b in bullets]

    wanted = ["- Main concern:", "- Nurse actions:", "- When to escalate:", "- Next step:"]
    picked = []

    for w in wanted:
        # find bullet with same heading
        found = None
        w_low = w.lower()
        for b in bullets:
            if b.lower().startswith(w_low):
                found = b
                break
        if found:
            picked.append(found)

    if len(picked) == 4:
        final = "\n".join(picked)
    else:
        # if not found, fallback to first 4 bullets
        final = "\n".join(bullets[:4]) if len(bullets) >= 4 else text

    if "DISCLAIMER" not in final.upper():
        final += f"\n\nDISCLAIMER: {DISCLAIMER}"
    return final

def _fallback_b(symptoms_str):
    return (
        f"- Main concern: Screening suggests an acute issue requiring assessment based on: {symptoms_str}.\n"
        "- Nurse actions: Check Temp/HR/RR/BP/SpO2; ask onset/duration/severity; assess hydration; document.\n"
        "- When to escalate: Chest pain, shortness of breath, confusion, fainting, unstable vitals, or rapid worsening.\n"
        "- Next step: Follow protocol, re-assess soon, and escalate for clinician review if uncertain.\n\n"
        f"DISCLAIMER: {DISCLAIMER}"
    )

def _generate_model_b(user_text, symptoms, top_pairs, note, strategy_name):
    symptoms_str = ", ".join(symptoms) if symptoms else "none detected"

    # dataset grounding top-2
    data_lines = []
    for disease, prob in top_pairs[:2]:
        desc, prec = _get_desc_prec(disease)
        desc = (desc[:220] if desc else "—")
        prec = (prec[:160] if prec else "—")
        data_lines.append(f"- {disease} (conf={prob:.2f}) | precautions: {prec} | desc: {desc}")
    data_ctx = "\n".join(data_lines) if data_lines else "No dataset candidates."

    # book grounding
    book_ctx = _rag_book_context(user_text, symptoms, k=3) or "No book snippets."

    rag_context = f"{data_ctx}\n\n{book_ctx}\n\nSafety note: {note}"

    template = PROMPT_BANK.get(strategy_name, PROMPT_S3_SAFETY_BIAS)
    prompt = _safe_format(template, symptoms_str, rag_context)

    gen = _pipe_generate(prompt)

    if _looks_bad(gen):
        return _fallback_b(symptoms_str)

    return _enforce_4_bullets(gen)

# =========================================================
# 4) Model A formatter (top1 red others green)
# =========================================================
def _format_model_a(top_pairs):
    a_out = ""
    for rank, (disease, prob) in enumerate(top_pairs, 1):
        color = "#c62828" if rank == 1 else "#2e7d32"
        desc, prec = _get_desc_prec(disease)
        desc = (desc.strip()[:450] + "…") if desc and desc.strip() else "—"
        prec = (prec.strip()[:250] + "…") if prec and prec.strip() else "—"

        a_out += f'<div style="color:{color}; font-weight:900;">{rank}. 🩺 Disease: {disease}</div>'
        a_out += f'<div style="color:{color}; font-weight:800;">Confidence: {prob:.2f}</div>'
        a_out += f'<div><b>Description:</b> {desc}</div>'
        a_out += f'<div><b>Precautions:</b> {prec}</div>'
        a_out += '<div style="border-bottom:1px dashed #d1d5db; margin:10px 0;"></div>'
    return a_out or "(no predictions)"

# =========================================================
# 5) Escalation note
# =========================================================
def _safety_note(user_text, symptoms, top_prob):
    escal_msgs = []
    red_flags = [
        "chest pain", "difficulty breathing", "shortness of breath",
        "unconscious", "seizure", "suicide", "self-harm"
    ]
    if any(rf in (user_text or "").lower() for rf in red_flags):
        escal_msgs.append("Potential emergency red-flag — escalate immediately per protocol")

    if not escal_msgs:
        if top_prob < 0.35:
            escal_msgs.append("Low confidence — gather more symptoms / clinician review")
        if len(symptoms) < 3:
            escal_msgs.append("Few symptoms — clarify onset/duration/severity and associated factors")

    if escal_msgs:
        return "ESCALATION RECOMMENDED:\n" + "\n".join(f"• {m}" for m in escal_msgs) + f"\n\n{DISCLAIMER}"
    return f"No immediate escalation. Monitor and reassess.\n\n{DISCLAIMER}"

# =========================================================
# 6) run_ui (safe)
# =========================================================
def run_ui(user_text, strategy_name):
    try:
        required = ["extract_symptoms", "symptom_vocab", "symptom_index", "lr", "le", "pipe"]
        missing = [x for x in required if x not in globals()]
        if missing:
            msg = "Developer hint: missing -> " + ", ".join(missing)
            return "none detected", "(Model A unavailable)", msg, msg

        symptoms = extract_symptoms(user_text) or []
        symptoms_str = ", ".join(symptoms) if symptoms else "none detected"

        if not symptoms:
            b_out = _fallback_b(symptoms_str)
            return symptoms_str, "(no matching symptoms)", b_out, _safety_note(user_text, symptoms, 0.0)

        vec = np.zeros(len(symptom_vocab), dtype=float)
        for s in symptoms:
            if s in symptom_index:
                vec[symptom_index[s]] = 1.0
        vec = vec.reshape(1, -1)

        probs = lr.predict_proba(vec)[0]
        top_idx = np.argsort(probs)[::-1][:5]
        top_pairs = [(le.inverse_transform([i])[0], float(probs[i])) for i in top_idx]

        top_prob = top_pairs[0][1] if top_pairs else 0.0
        note = _safety_note(user_text, symptoms, top_prob)

        a_out = _format_model_a(top_pairs)
        b_out = _generate_model_b(user_text, symptoms, top_pairs, note, strategy_name)

        return symptoms_str, a_out, b_out, note

    except Exception as e:
        msg = f"Developer error: {type(e).__name__}: {e}"
        return "none detected", "(error)", msg, msg

# =========================================================
# 7) UI
# =========================================================
with gr.Blocks(theme=gr.themes.Soft(primary_hue="blue", secondary_hue="green"), title="Nurse Screening Assistant") as demo:
    gr.Markdown("# 🩺 Medical Screening Assistant")
    gr.Markdown("**Educational prototype for nurses** – Symptom screening support only")

    gr.Markdown(
        f"<div style='background-color:#ffebee; color:#c62828; font-weight:bold; font-size:16px; padding:16px; "
        f"border-radius:8px; border:2px solid #c62828; margin-bottom:20px;'>{DISCLAIMER}</div>"
    )

    strategy = gr.Dropdown(
        choices=list(PROMPT_BANK.keys()),
        value="S3: Safety + Bias (Best)",
        label="Model B Prompt Strategy"
    )

    inp = gr.Textbox(
        lines=6,
        label="Patient Symptoms Description",
        placeholder="e.g. high fever for 4 days, dry cough, severe fatigue, muscle pain"
    )
    btn = gr.Button("Analyze", variant="primary")

    with gr.Row(equal_height=True):
        with gr.Column():
            out_sym = gr.Textbox(label="Extracted Symptoms", lines=3)
            out_a = gr.Markdown(label="Model A – Logistic Regression (Top Predictions)")
        with gr.Column():
            out_b = gr.Textbox(lines=10, label="Model B – Nurse Guidance (4 bullets)")
            out_note = gr.Textbox(label="Safety & Escalation Note", lines=6, interactive=False)

    btn.click(run_ui, inputs=[inp, strategy], outputs=[out_sym, out_a, out_b, out_note])

demo.launch(share=True, debug=True)


/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.42.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://19131045232f37f4d8.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://19131045232f37f4d8.gradio.live
